In [ ]:
import os
import gc
import time
from collections import Counter

import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision.models.video import r3d_18, R3D_18_Weights
import torchvision.transforms as T
from sklearn.model_selection import train_test_split

# --- 1. KAGGLE CONFIGURATION ---
# Read-only dataset provided by Kaggle
BASE_DIR = "/kaggle/input/datasets/argrand/surgbench-e-phase-classification/phase_classification_dataset"

# Writable directory for saving models
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- 2. DATA VERIFICATION ---
def check_local_distribution():
    if not os.path.exists(BASE_DIR):
        print(f"Directory {BASE_DIR} does not exist. Check the Kaggle input path!")
        return
        
    print(f"\n{'Phase Folder':<45} | {'File Count':<10}")
    print("-" * 60)
    total_files = 0
    folders = [f for f in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, f))]
    
    for folder in sorted(folders):
        folder_path = os.path.join(BASE_DIR, folder)
        count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
        print(f"{folder:<45} | {count:<10}")
        total_files += count
    print("-" * 60)
    print(f"Total files in dataset: {total_files}\n")

check_local_distribution()

In [ ]:
# --- BLOCK 2: CORRUPTION SCAN + DATASET CLASS & DATALOADERS ---
import os
import json
import cv2
import torch
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from collections import Counter
from multiprocessing import Pool
from tqdm import tqdm

# Kaggle paths: BASE_DIR should already point at your dataset,
# e.g. /kaggle/input/<dataset-name>/... (read-only) or /kaggle/working/... (writable)
# Scan results are cached to /kaggle/working since /kaggle/input is read-only.
SCAN_CACHE_PATH = "/kaggle/working/video_scan_results.json"

def check_video(filepath):
    """Checks if a video file is readable and has valid frames."""
    if os.path.getsize(filepath) < 1024:
        return filepath, "File size too small (< 1KB)"
    try:
        cap = cv2.VideoCapture(filepath)
        if not cap.isOpened():
            cap.release()
            return filepath, "OpenCV cannot open file"
        ret, _ = cap.read()
        cap.release()
        if not ret:
            return filepath, "Cannot read first frame"
        return filepath, "OK"
    except Exception as e:
        return filepath, f"Exception: {str(e)}"

def run_verification(base_dir, cache_path=SCAN_CACHE_PATH, force_rescan=False, num_workers=8):
    # Reuse cached results if available, so this only ever runs once per dataset version
    if os.path.exists(cache_path) and not force_rescan:
        print(f"Loading cached scan results from {cache_path}")
        with open(cache_path) as f:
            data = json.load(f)
        print(f"✅ Valid: {len(data['valid'])}  ❌ Corrupted: {len(data['corrupted'])}")
        return data["valid"], data["corrupted"]

    print("Gathering files...")
    all_videos = []
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.endswith(('.mp4', '.avi')):
                all_videos.append(os.path.join(root, f))

    print(f"Found {len(all_videos)} video files. Verifying integrity...")

    with Pool(num_workers) as p:
        results = list(tqdm(p.imap(check_video, all_videos), total=len(all_videos)))

    valid = [fp for fp, status in results if status == "OK"]
    corrupted = [(fp, status) for fp, status in results if status != "OK"]

    print("\n--- Verification Complete ---")
    if not corrupted:
        print("✅ All videos are completely valid and readable!")
    else:
        print(f"❌ Found {len(corrupted)} corrupted files:")
        for path, error in corrupted:
            print(f" - {os.path.basename(path)}: {error}")

    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, "w") as f:
        json.dump({"valid": valid, "corrupted": corrupted}, f, indent=2)
    print(f"💾 Scan results cached to {cache_path}")

    return valid, corrupted


class VideoDataset(Dataset):
    def __init__(self, folders, root_dir, valid_files=None):
        self.files = []
        self.labels = []
        self.resize = T.Resize((112, 112), antialias=True)
        self.class_to_idx = {name: i for i, name in enumerate(sorted(folders))}
        valid_set = set(valid_files) if valid_files is not None else None
        for folder in folders:
            path = os.path.join(root_dir, folder)
            if not os.path.isdir(path):
                continue
            label = self.class_to_idx[folder]
            for f in os.listdir(path):
                if f.endswith(('.mp4', '.avi')):
                    fpath = os.path.join(path, f)
                    if valid_set is not None and fpath not in valid_set:
                        continue  # skip known-corrupt files
                    self.files.append(fpath)
                    self.labels.append(label)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        cap = cv2.VideoCapture(file_path)
        frames = []
        try:
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames < 1:
                cap.release()
                return torch.zeros((3, 16, 112, 112)), -1
            step = max(1, total_frames // 16)
            frame_count = 0
            while len(frames) < 16:
                ret, frame = cap.read()
                if not ret:
                    break
                if frame_count % step == 0:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(torch.from_numpy(frame))
                frame_count += 1
            cap.release()
            if len(frames) == 0:
                return torch.zeros((3, 16, 112, 112)), -1
            video = torch.stack(frames)
            if video.shape[0] < 16:
                padding = video[-1].repeat(16 - video.shape[0], 1, 1, 1)
                video = torch.cat([video, padding], dim=0)
            else:
                video = video[:16]
            video = video.permute(0, 3, 1, 2).float() / 255.0
            video = self.resize(video).permute(1, 0, 2, 3)
            return video, self.labels[idx]
        except Exception:
            if cap.isOpened():
                cap.release()
            return torch.zeros((3, 16, 112, 112)), -1


def collate_fn(batch):
    batch = [b for b in batch if b[1] != -1]
    return torch.utils.data.dataloader.default_collate(batch) if batch else (torch.empty(0), torch.empty(0))


# --- RUN SCAN, THEN BUILD DATASET FROM CLEAN FILE LIST ---
valid_files, corrupted_files = run_verification(BASE_DIR)

folders = sorted([f for f in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, f)) and not f.startswith('.')])
dataset = VideoDataset(folders, BASE_DIR, valid_files=valid_files)
print(f"Total dataset size: {len(dataset)} videos mapped across {len(folders)} classes.")

train_idx, test_idx = train_test_split(range(len(dataset)), test_size=0.2, random_state=42)
label_counts = Counter([dataset.labels[i] for i in train_idx])
weights = [1.0 / label_counts[dataset.labels[i]] for i in train_idx]
sampler = WeightedRandomSampler(weights, num_samples=len(train_idx), replacement=True)

train_loader = DataLoader(Subset(dataset, train_idx), batch_size=8, sampler=sampler, collate_fn=collate_fn, pin_memory=True, num_workers=2)
test_loader = DataLoader(Subset(dataset, test_idx), batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=2)

In [ ]:
# --- BLOCK 3: MODEL SETUP & CHECKPOINT LOADING (OFFLINE MODE) ---

# 1. Path to the PyTorch r3d_18 backbone weights you uploaded
OFFLINE_BACKBONE_WEIGHTS = "/kaggle/input/datasets/argrand/r3d18-pre-defined/r3d_18-b3b3357e.pth"

# 2. Path to your previously trained checkpoint
UPLOADED_PTH_PATH = "/kaggle/input/datasets/argrand/surgbench-e-phase-classification/r3d18_ep7.pth"
# CRITICAL: weights=None prevents PyTorch from trying to connect to the internet
model = r3d_18(weights=None)

# --- Load the Offline Backbone Weights ---
if os.path.exists(OFFLINE_BACKBONE_WEIGHTS):
    print(f"✅ Loading offline backbone weights from {OFFLINE_BACKBONE_WEIGHTS}")
    model.load_state_dict(torch.load(OFFLINE_BACKBONE_WEIGHTS, map_location=device))
else:
    print(f"❌ ERROR: Offline weights not found at {OFFLINE_BACKBONE_WEIGHTS}.")

# Freeze the backbone initially
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer for our specific number of classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(folders))
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
start_epoch = 0

# --- Check for Previous Training Checkpoints (Resume Logic) ---
existing_checkpoints = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("r3d18_ep") and f.endswith(".pth")]

if existing_checkpoints:
    latest_cp = max(existing_checkpoints, key=lambda x: int(x.split('ep')[1].split('.pth')[0]))
    target_checkpoint = os.path.join(CHECKPOINT_DIR, latest_cp)
elif os.path.exists(UPLOADED_PTH_PATH):
    target_checkpoint = UPLOADED_PTH_PATH
else:
    target_checkpoint = None

if target_checkpoint:
    print(f"🔄 Loading saved progress checkpoint: {target_checkpoint}")
    checkpoint = torch.load(target_checkpoint, map_location=device)
    start_epoch = checkpoint['epoch'] + 1
    
    state_dict = checkpoint['model_state_dict']
    is_perfect_match = True

    # 1. Handle Model Weights
    if state_dict['fc.weight'].shape[0] != len(folders):
        print(f"⚠️ Class count mismatch! Checkpoint has {state_dict['fc.weight'].shape[0]} classes, but dataset has {len(folders)}.")
        print("Dropping the final classification layer weights to adapt to the new dataset size...")
        
        state_dict.pop('fc.weight', None)
        state_dict.pop('fc.bias', None)
        
        model.load_state_dict(state_dict, strict=False)
        nn.init.xavier_uniform_(model.fc.weight)
        nn.init.zeros_(model.fc.bias)
        
        is_perfect_match = False # Flag so we don't try to load the old optimizer
    else:
        model.load_state_dict(state_dict)

    # 2. Build Optimizer Structure Based on Epoch
    if start_epoch >= 3:
        print("[INFO] Resuming in unfrozen state. Setting up full optimizer structure...")
        for param in model.parameters(): param.requires_grad = True
        optimizer = optim.Adam([
            {'params': model.stem.parameters(), 'lr': 1e-5},
            {'params': model.layer1.parameters(), 'lr': 1e-5},
            {'params': model.layer2.parameters(), 'lr': 1e-5},
            {'params': model.layer3.parameters(), 'lr': 1e-5},
            {'params': model.layer4.parameters(), 'lr': 1e-5},
            {'params': model.fc.parameters(), 'lr': 1e-4}
        ])
    else:
        optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

    # 3. Load Optimizer State (Memory) ONLY if shapes matched
    if is_perfect_match:
        try: 
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            print("✅ Optimizer memory loaded successfully.")
        except ValueError: 
            print("⚠️ Optimizer param mismatch. Starting Adam fresh.")
    else:
        print("⚠️ Optimizer starting fresh due to architecture change.")

    print(f"✅ Successfully loaded! Resuming from Epoch {start_epoch + 1}")

else:
    print("⚠️ No prior training checkpoint found. Initializing head with random weights.")
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)
    optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

In [ ]:
num_epochs = 50

for epoch in range(start_epoch, num_epochs):
    print(f"\n{'='*40}\n🚀 Starting Epoch {epoch+1}/{num_epochs}")

    if epoch == 3:
        print("[INFO] Unfreezing the backbone for fine-tuning...")
        for param in model.parameters(): param.requires_grad = True
        optimizer = optim.Adam([
            {'params': model.stem.parameters(), 'lr': 1e-5},
            {'params': model.layer1.parameters(), 'lr': 1e-5},
            {'params': model.layer2.parameters(), 'lr': 1e-5},
            {'params': model.layer3.parameters(), 'lr': 1e-5},
            {'params': model.layer4.parameters(), 'lr': 1e-5},
            {'params': model.fc.parameters(), 'lr': 1e-4}
        ])

    model.train()
    running_loss = 0.0
    start_time = time.time()

    for batch_idx, (x, y) in enumerate(train_loader):
        if x.numel() == 0: continue
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (batch_idx + 1) % 20 == 0:
            print(f"   Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    # --- EVALUATION ---
    model.eval()
    correct = total = 0
    class_correct = Counter()
    class_total = Counter()

    print("\nEvaluating on Test Set...")
    with torch.no_grad():
        for x, y in test_loader:
            if x.numel() == 0: continue
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

            for i in range(len(y)):
                label = y[i].item()
                class_total[label] += 1
                if predicted[i] == label:
                    class_correct[label] += 1

    epoch_acc = correct / total if total > 0 else 0.0
    avg_loss = running_loss / len(train_loader) if len(train_loader) > 0 else 0.0
    time_elapsed = time.time() - start_time

    print(f"--- Epoch {epoch+1} Summary ---")
    print(f"Time: {time_elapsed//60:.0f}m {time_elapsed%60:.0f}s")
    print(f"Total Train Loss: {avg_loss:.4f} | Overall Test Accuracy: {epoch_acc:.2%}")

    if epoch_acc < 0.85:
        print("Per-Class Accuracy Breakdown:")
        for label in sorted(class_total.keys()):
            acc = class_correct[label] / class_total[label] if class_total[label] > 0 else 0
            print(f"   -> Class {label:<2} (Total: {class_total[label]:<3}): {acc:.2%}")

    # --- CHECKPOINTING ---
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f"r3d18_ep{epoch+1}.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
        'val_acc': epoch_acc
    }, checkpoint_path)
    print(f"💾 Checkpoint saved to Output space: {checkpoint_path}")

    gc.collect()
    torch.cuda.empty_cache()

print("\n🎉 Training Complete!")